In [2]:
%pip install ta scikit-learn pandas numpy matplotlib mplfinance torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.1 MB/s eta 0:00:00
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=6b3c09a0630c2e7ce3850e0ca2183e14ea892e4c3602cdc130e1ff08648d016d
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [ ]:
import pandas as pd
import numpy as np
import os
import ta
from sklearn.preprocessing import StandardScaler

# --- הגדרת המטבע ---
SYMBOL = 'BTCUSDT'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/CryptoProject'
except ImportError:
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()

SEQ_LENGTH = 60
PREDICT_AHEAD = 1        # predict %B 1 step (5m) ahead - matches the working models
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
STEP = 5

FEATURE_COLS = [
    'log_ret', 'rsi', 'rsi_change', 'rsi_accel', 'macd', 'macd_slope',
    'bb_pband_change', 'volume', 'ma_dist', 'volume_z', 'vol_spike', 'adx',
    'hour_sin', 'hour_cos', 'mom_3', 'mom_5'
]

# המטרה עבור Transformer
TARGET_COL = 'bb_pband'

class TransformerDataPreprocessor:
    def load_and_clean_data(self, filepath):
        print(f"Loading data from {filepath}...")
        df = pd.read_csv(filepath)
        df['open_time'] = pd.to_datetime(df['open_time'])
        df = df.sort_values('open_time').drop_duplicates(subset=['open_time']).ffill().dropna()

        # Correct log return (no buggy "* 10" inside the log)
        df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
        df['rsi'] = ta.momentum.rsi(df['close'], window=14) / 100.0
        df['rsi_change'] = df['rsi'].diff(periods=3)
        df['rsi_accel'] = df['rsi_change'].diff(periods=2)

        macd = ta.trend.MACD(df['close'])
        macd_raw = macd.macd_diff()
        df['macd'] = (macd_raw - macd_raw.rolling(window=100).mean()) / (macd_raw.rolling(window=100).std() + 1e-9)
        df['macd_diff'] = macd.macd_diff()
        df['macd_slope'] = df['macd_diff'].diff(periods=2)

        bb = ta.volatility.BollingerBands(df['close'], window=20, window_dev=2)
        df['bb_pband'] = bb.bollinger_pband()
        df['bb_pband_change'] = df['bb_pband'].diff(periods=1)

        df['volume'] = np.log(df['volume'] + 1)
        df['volume'] = (df['volume'] - df['volume'].mean()) / (df['volume'].std() + 1e-9)
        df['vol_ma'] = df['volume'].rolling(window=20).mean()
        df['vol_std'] = df['volume'].rolling(window=20).std()
        df['volume_z'] = (df['volume'] - df['vol_ma']) / (df['vol_std'] + 1e-9)
        df['vol_spike'] = (df['volume'] > (df['vol_ma'] * 2)).astype(float)

        df['ma_20'] = df['close'].rolling(window=20).mean()
        df['ma_dist'] = (df['close'] - df['ma_20']) / (df['ma_20'] + 1e-9) * 10.0

        adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'], window=14)
        df['adx'] = adx.adx()

        df['hour'] = df['open_time'].dt.hour
        df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
        df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

        # Enhanced momentum features
        df['mom_3'] = df['close'] / df['close'].shift(3) - 1
        df['mom_5'] = df['close'] / df['close'].shift(5) - 1

        return df.dropna()

    def create_sequences(self, data, target, seq_length, step=1):
        xs, ys = [], []
        # Predict %B PREDICT_AHEAD steps after the end of the sequence.
        for i in range(0, len(data) - seq_length - PREDICT_AHEAD + 1, step):
            xs.append(data[i : (i + seq_length)])
            ys.append(target[i + seq_length + PREDICT_AHEAD - 1])
        return np.array(xs), np.array(ys)

    def process(self):
        data_path = os.path.join(BASE_DIR, 'data', f'{SYMBOL}_5m_data.csv')
        df = self.load_and_clean_data(data_path)

        data = df[FEATURE_COLS].values
        target = df[TARGET_COL].values

        n = len(data)
        train_end = int(n * TRAIN_SPLIT)
        val_end = int(n * (TRAIN_SPLIT + VAL_SPLIT))

        scaler = StandardScaler()
        train_data_scaled = scaler.fit_transform(data[:train_end])
        val_data_scaled = scaler.transform(data[train_end:val_end])
        test_data_scaled = scaler.transform(data[val_end:])

        X_train, y_train = self.create_sequences(train_data_scaled, target[:train_end], SEQ_LENGTH, STEP)
        X_val, y_val = self.create_sequences(val_data_scaled, target[train_end:val_end], SEQ_LENGTH, STEP)
        X_test, y_test = self.create_sequences(test_data_scaled, target[val_end:], SEQ_LENGTH, STEP)

        save_dir = os.path.join(BASE_DIR, 'processed_data_transformer', SYMBOL)
        os.makedirs(save_dir, exist_ok=True)

        np.save(os.path.join(save_dir, 'X_train.npy'), X_train)
        np.save(os.path.join(save_dir, 'y_train.npy'), y_train)
        np.save(os.path.join(save_dir, 'X_val.npy'), X_val)
        np.save(os.path.join(save_dir, 'y_val.npy'), y_val)
        np.save(os.path.join(save_dir, 'X_test.npy'), X_test)
        np.save(os.path.join(save_dir, 'y_test.npy'), y_test)
        print(f"✅ Preprocessing Complete for {SYMBOL} (Transformer, %B +1 step). Saved to {save_dir}")

if __name__ == "__main__":
    TransformerDataPreprocessor().process()
